# Downstream CNN — recovered spectra

Trains the multi-task CNN (variant classification plus concentration regression)
on recovered spectra, then evaluates it on the held-out test set and on the
independent unknown-concentration test set.

**Loss weighting** — `CLASS_WEIGHT : REG_WEIGHT = 100 : 1`, the ratio reported in
the manuscript, selected by grid search over nine decade-spaced ratios.

**Input** — the combined recovered-spectra CSVs from
`../2_sequential_dilution/03_merge_extracted.ipynb`, and the per-sample unknown
test folders from `02_recover_unknown_test_samples.ipynb`.

**Output** — trained model, loss curves, confusion matrices, per-concentration
accuracy, and per-variant regression tables.

**Feeds** — Fig. 6 and Supplementary Figs. S16–S21.

**Note** — the unknown test set is scored at sample level: a sample is assigned
to a class only when at least 70% of its spectra agree.

**Counterpart** — `04_train_eval_hybridized.ipynb` is the same architecture
trained on hybridized spectra, and the two differ only in their input and their
loss weighting.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import math
import ast
import os

In [ ]:
from sklearn.metrics import mean_absolute_error,accuracy_score,confusion_matrix,r2_score

from keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten, Conv1D, MaxPool1D,BatchNormalization,Dropout
from tensorflow.keras.models import Model
from keras.optimizers import Adam
from tensorflow import keras
import tensorflow as tf

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils import shuffle
from tqdm import trange

In [ ]:
tf.__version__, 

In [ ]:
from platform import python_version

print(python_version())

In [ ]:
# --- MUST run before importing matplotlib ---
import os
print("MPLBACKEND (before):", os.environ.get("MPLBACKEND"))
os.environ.pop("MPLBACKEND", None)   # remove any forced backend like 'Agg'

from IPython import get_ipython
ip = get_ipython()
if ip:
    ip.run_line_magic("matplotlib", "inline")

import matplotlib
import matplotlib.pyplot as plt

print("Matplotlib version:", matplotlib.__version__)
print("Active backend:    ", matplotlib.get_backend())
print("matplotlibrc:      ", matplotlib.matplotlib_fname())

fig, ax = plt.subplots()
ax.plot([0, 1, 2, 3], [0, 1, 0, 1])
ax.set_title("Inline display test")
plt.show()

##### Move data.

In [ ]:
SCRATCH_WORKING_PATH = "/scratch/jc76425/DNA_RNA_hybridization/results/03092026-combination_of_8_extracted_viruses_and_DNA"

INFERENCE_MODE = True

DATE = "03092026"
CLASS_WEIGHT = 100  # published recovered ratio w1:w2 = 100:1 
REG_WEIGHT = 1
RESULT_PATH = f"/scratch/jc76425/DNA_RNA_hybridization/results/{DATE}-extracted_data-500_epochs_weight_{CLASS_WEIGHT}vs{REG_WEIGHT}"
os.makedirs(RESULT_PATH, exist_ok=True)

##### Load data.

In [ ]:
WORKING_PATH = f"{SCRATCH_WORKING_PATH}"

In [ ]:
Mix_dat = pd.read_csv(WORKING_PATH + "/03092026-all_extracted_spectra_combined.csv")

In [ ]:
Mix_dat

In [ ]:
labels = np.unique(Mix_dat['Label'])
concs = np.unique(Mix_dat['Conc'])
print(labels,concs)

In [ ]:
print(Mix_dat.groupby(['Label','Conc'], group_keys=False).apply(lambda x: x.shape))

#### Subset: 20% of total data.

1. Data splitting.

In [ ]:
subset = Mix_dat
SEED = 42  # choose any integer, save it to reuse

## train, validation, test: 7/1.5/1.5

tra_val = subset.groupby(['Label','Conc'], group_keys=False).apply(lambda x: x.sample(math.ceil(x.shape[0]*0.85), random_state=SEED))
test = subset.loc[subset.index.difference(tra_val.index)]

train = tra_val.groupby(['Label','Conc'], group_keys=False).apply(lambda x: x.sample(math.ceil(x.shape[0]*7/8.5), random_state=SEED))
val = tra_val.loc[tra_val.index.difference(train.index)]

In [ ]:
train = pd.read_csv(WORKING_PATH + "/03092026-all_train_extracted_spectra_combined.csv")
val = pd.read_csv(WORKING_PATH + "/03092026-all_val_extracted_spectra_combined.csv")
test = pd.read_csv(WORKING_PATH + "/03092026-all_test_extracted_spectra_combined.csv")
train.shape,val.shape,test.shape,

In [ ]:
test

In [ ]:
def transform_label(label):
    # Check for condition 1: Replace specific strings with 'Reference'
    if label in (["['AF']", "['AgNR@SiO2']", "['DMEM']", "[]"]):
        return "['Reference']"
    
    # Check for condition 2: Replace 'COVNL63' or 'CoVNL63' with 'CovNL63'
    if 'COVNL63' in label or 'CovNL63' in label:
        label = label.replace('COVNL63', 'CoVNL63')
        label = label.replace('CoVNL63', 'CoVNL63')
        label = label.replace('CovNL63', 'CoVNL63')
    
    return label


# Apply the custom function to the "Label" column
train['Label'] = train['Label'].apply(transform_label)
val['Label'] = val['Label'].apply(transform_label)
test['Label'] = test['Label'].apply(transform_label)

2. Input X.

In [ ]:
## X
X_train = train.iloc[:,:-2].values
X_test = test.iloc[:,:-2].values
X_val = val.iloc[:,:-2].values

## standardize X
scaler1 = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = scaler1.transform(X_train)

scaler2 = preprocessing.StandardScaler().fit(X_test)
X_test_scaled = scaler2.transform(X_test)

scaler3 = preprocessing.StandardScaler().fit(X_val)
X_val_scaled = scaler3.transform(X_val)

sample_size_train = X_train_scaled.shape[0]
sample_size_test = X_test_scaled.shape[0]
sample_size_val = X_val_scaled.shape[0]
print(sample_size_train)

time_steps_train = X_train_scaled.shape[1]
time_steps_test = X_test_scaled.shape[1]
time_steps_val = X_val_scaled.shape[1]
print(time_steps_train)

input_dimension = 1


## reshape the dataset
X_train_reshape = X_train_scaled.reshape(sample_size_train, time_steps_train, input_dimension)
X_test_reshape = X_test_scaled.reshape(sample_size_test, time_steps_test, input_dimension)
X_val_reshape = X_val_scaled.reshape(sample_size_val, time_steps_val, input_dimension)

3. One-hot encoding for y.

In [ ]:
## convert string to list

# training
label1 = train['Label'].values
conc1  = train['Conc'].values

label_train = []
conc_train = []
for i in range(train.shape[0]):
    a = ast.literal_eval(label1[i])
    b = ast.literal_eval(conc1[i])
    
    label_train.append(a)
    conc_train.append(b)
    
# test   
label2 = test['Label'].values
conc2  = test['Conc'].values

label_test = []
conc_test = []
for i in range(test.shape[0]):
    a = ast.literal_eval(label2[i])
    b = ast.literal_eval(conc2[i])
    
    label_test.append(a)
    conc_test.append(b)
    
# val   
label3 = val['Label'].values
conc3  = val['Conc'].values

label_val = []
conc_val = []
for i in range(val.shape[0]):
    a = ast.literal_eval(label3[i])
    b = ast.literal_eval(conc3[i])
    
    label_val.append(a)
    conc_val.append(b)

In [ ]:
## classification
# train
mlb1 = MultiLabelBinarizer()
y_train_class = mlb1.fit_transform(list(label_train))

y_train_class = pd.DataFrame(y_train_class)
y_train_class.columns = mlb1.classes_
y_train_class

In [ ]:
## classification
# test
mlb2 = MultiLabelBinarizer()
y_test_class = mlb2.fit_transform(list(label_test))

y_test_class = pd.DataFrame(y_test_class)
y_test_class.columns = mlb2.classes_
y_test_class

In [ ]:
## classification
# val
mlb3 = MultiLabelBinarizer()
y_val_class = mlb3.fit_transform(list(label_val))

y_val_class = pd.DataFrame(y_val_class)
y_val_class.columns = mlb3.classes_
y_val_class

In [ ]:
## regression
# train
y_train_reg = np.zeros(y_train_class.shape)

for i in range(y_train_class.shape[0]):
    y_train_reg[i][(y_train_class.values)[i]==1]=np.log10([x + 1e-20 for x in conc_train[i]])
    
                
y_train_reg = pd.DataFrame(y_train_reg)
y_train_reg.columns = mlb1.classes_
y_train_reg

In [ ]:
## regression
# test
y_test_reg = np.zeros(y_test_class.shape)

for i in range(y_test_class.shape[0]):
    y_test_reg[i][(y_test_class.values)[i]==1]=np.log10([x + 1e-20 for x in conc_test[i]])
    
                
y_test_reg = pd.DataFrame(y_test_reg)
y_test_reg.columns = mlb2.classes_
y_test_reg

In [ ]:
## regression
# val
y_val_reg = np.zeros(y_val_class.shape)

for i in range(y_val_class.shape[0]):
     y_val_reg[i][(y_val_class.values)[i]==1]=np.log10([x + 1e-20 for x in conc_val[i]])
    
                
y_val_reg = pd.DataFrame(y_val_reg)
y_val_reg.columns = mlb3.classes_
y_val_reg

#### Multi-task model.

1. One-step model.

In [ ]:
def MixNet_10_Conv(n_timesteps, n_inputs, n_outputs):
    # Input layer
    visible = Input(shape=(n_timesteps, n_inputs,))
    
    # Convolutional blocks
    filters = 64  # Starting number of filters
    hidden = visible
    for i in range(10):
        hidden = Conv1D(filters=filters, kernel_size=3, activation='relu', padding='same', name=f'Conv1D_{i+1}')(hidden)
        hidden = BatchNormalization(name=f"BatchNorm_{i+1}")(hidden)
        if i < 10 - 1:  # Add pooling except for the last convolutional layer
            hidden = MaxPool1D(pool_size=2, strides=2, name=f'MaxPooling1D_{i+1}')(hidden)
        if i < 7:  # Increase the number of filters for the first 8 blocks
            filters *= 2
        if filters > 512:  # Limit the maximum number of filters
            filters = 512

    # Flatten and Dense layers
    hidden = Flatten()(hidden)
    hidden = Dense(200, activation='relu', name='Dense_1')(hidden)
    hidden = Dense(100, activation='relu', name='Dense_2')(hidden)

    # Output layers
    out_reg = Dense(n_outputs, activation='linear', name="Reg")(hidden)
    out_clas = Dense(n_outputs, activation='sigmoid', name="Class")(hidden)


    # Define and compile the model
    model = Model(inputs=visible, outputs=[out_clas, out_reg], name="MixNet_10_Conv")
    
    # Compile the model with appropriate metrics
    model.compile(optimizer='adam', 
                  loss={'Class': 'binary_crossentropy', 'Reg': 'mae'},
                  metrics={'Class': 'accuracy', 'Reg': 'mse'},
                  loss_weights={'Class': CLASS_WEIGHT, 'Reg': REG_WEIGHT})

    return model

2. model fitting.

In [ ]:
model = MixNet_10_Conv(time_steps_train,1,np.unique(label1).shape[0])
model.summary()

In [ ]:
# fit the keras model on the dataset
if not INFERENCE_MODE:
    
    history=model.fit(X_train_reshape, [y_train_class,y_train_reg], epochs=500, 
                      batch_size=32, verbose=1,validation_data=(X_val_reshape,[y_val_class,y_val_reg]))

In [ ]:
if not INFERENCE_MODE:
    loss2 = pd.DataFrame({"Class_loss":history.history["Class_loss"],
                            "val_Class_loss":history.history["val_Class_loss"],
                           "Reg_loss": history.history["Reg_loss"],
                            "val_Reg_loss": history.history["val_Reg_loss"]})
    loss2

In [ ]:
if not INFERENCE_MODE:
    loss_df = loss2
    loss_df

In [ ]:
if not INFERENCE_MODE:
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.plot(loss_df["Class_loss"].values, label="Training")
    plt.plot(loss_df["val_Class_loss"].values, label="Validation")
    plt.xlabel("Epochs")
    plt.ylabel("Classification loss")
    plt.title('Classification')
    plt.legend()
    
    plt.subplot(122)
    plt.plot(loss_df["Reg_loss"].values, label="Training")
    plt.plot(loss_df["val_Reg_loss"].values, label="Validation")
    plt.xlabel("Epochs")
    plt.ylabel("Regression loss")
    plt.title('Regression')
    plt.legend()

    plt.savefig(RESULT_PATH + "/loss_curves.png")
    
    plt.show()

In [ ]:
if not INFERENCE_MODE:
    pd.DataFrame(loss_df).to_csv(RESULT_PATH + "/Loss_500_Epoch.csv", index=False)

In [ ]:
# save model to file
if not INFERENCE_MODE:
    model.save(RESULT_PATH + '/Model_500_Epoch.h5')

3. Prediction in test.

In [ ]:
if INFERENCE_MODE:
    model = tf.keras.models.load_model(RESULT_PATH + '/Model_500_Epoch.h5', compile=False)
    model

In [ ]:
y_hat_class, y_hat_reg = model.predict(X_test_reshape)

# evaluate accuracy for classification model
y_hat_class = y_hat_class.round()
acc = accuracy_score(y_test_class, y_hat_class)
print('Accuracy: %.3f' % acc)


# calculate error for regression model
error = np.abs(y_test_reg-y_hat_reg).sum()/y_hat_reg.shape[0]

print(error)

#### Classification

In [ ]:
## convert 0-1 label to string

# true label
y_true_label = test['Label'].values

# predicted label
classes = np.unique(y_test_class.columns)

y_pred_label = []
for i in range(y_hat_class.shape[0]):
    ind = np.where(y_hat_class[i]==1)
    label = list(classes[ind])
    y_pred_label.append(label)

y_pred_label = np.array([str(j) for j in y_pred_label])

In [ ]:
## classification
print(round(accuracy_score(y_true_label, y_pred_label), 4))

index =np.unique(y_true_label)
# Sort the labels by length and then alphabetically
index = sorted(index, key=lambda x: (x.count(','), x))

cm = confusion_matrix(y_true_label, y_pred_label, labels=index)
cm_df = pd.DataFrame(cm,index,index)  

# Draw the confusion matrix with borders
plt.figure(figsize=(20, 20))
ax = sns.heatmap(cm_df, annot=False, fmt="d", cmap='Blues_r', linewidths=2, linecolor='black')

def get_text_color(val, min_val, max_val):
    """Determine the text color based on the cell value."""
    mid_val = (max_val - min_val) / 2
    if val < mid_val:
        return 'white'
    else:
        return 'black'    
    
# Manually add annotations for non-zero values
rows, cols = cm.shape
min_val, max_val = np.min(cm), np.max(cm)
for i in range(rows):
    for j in range(cols):
        if cm[i, j] != 0:
            text_color = get_text_color(cm[i, j], min_val, max_val)
            ax.text(j + 0.5, i + 0.5, cm[i, j], fontsize=14,
                    horizontalalignment='center',
                    verticalalignment='center', color=text_color)

plt.xlabel("Predicted label")
plt.ylabel("True label")
ax.collections[0].colorbar.remove()

if not INFERENCE_MODE:
    plt.savefig(RESULT_PATH + "/test_confusion matrix_500epochs.png")
plt.show()

In [ ]:
if not INFERENCE_MODE:
    pd.DataFrame(cm_df).to_csv(RESULT_PATH + "/test_confusion matrix_500epochs.csv",index=True)

In [ ]:
# Normalize along the row to get percentage
cm_percent = 100*cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_percent_df = pd.DataFrame(cm_percent, index, index)
# cm_percent_df
plt.figure(figsize=(10, 10))
ax = sns.heatmap(cm_percent_df, annot=False, fmt=".2%", cmap='Blues_r', linewidths=2, linecolor='black')

def get_text_color(val, min_val, max_val):
    mid_val = (1 - 0) / 2
    if val < mid_val:
        return 'white'
    else:
        return 'black'

rows, cols = cm_percent_df.shape
min_val, max_val = np.min(cm_percent_df), np.max(cm_percent_df)
for i in range(rows):
    for j in range(cols):
        if cm_percent_df.iloc[i, j] != 0:
            text_color = get_text_color(cm_percent_df.iloc[i, j], min_val, max_val)
            ax.text(j + 0.5, i + 0.5, f"{cm_percent[i, j]:.1f}",
                    fontsize=14, horizontalalignment='center',
                    verticalalignment='center', color=text_color)

plt.xlabel("Predicted label")
plt.ylabel("True label")
ax.collections[0].colorbar.remove()

if not INFERENCE_MODE:
    plt.savefig(RESULT_PATH + "/test_confusion matrix_percentage_500epochs.png")
plt.show()

In [ ]:
if not INFERENCE_MODE:
    pd.DataFrame(cm_percent_df).to_csv(RESULT_PATH + "/test_confusion matrix_percentage_500epochs.csv",index=True)

In [ ]:
# Every sample carries exactly one label, so this mask selects all rows.
# It is kept because the regression cells below index with it.
single_index = np.sum(y_test_class.values, axis = 1) == 1


#### misclassification

In [ ]:
## misclassification
mis_index = np.where(y_true_label != y_pred_label)
true_y = y_true_label[mis_index]
pred_y = y_pred_label[mis_index]
y_true_conc = test['Conc'].values
true_y_conc = y_true_conc[mis_index]
pred_y_conc_list = []  # Initialize an empty list to store the predicted concentrations

# Loop through each misclassified sample
for i, preds in enumerate(pred_y):
    sample_pred_y_conc = []
    preds_list = ast.literal_eval(preds)
    # Loop through each predicted label for the current misclassified sample
    for pred in preds_list:  # Assuming preds is a list or array
        # Find the index of the predicted label in 'index'
        pred_str = str([pred]) # should yield something like "['HMPVA']"
        pred_idx = [i for i, x in enumerate(index) if x == pred_str]

        # Check if a matching index was found
        if len(pred_idx) > 0:
            pred_idx = pred_idx[0]
        else:
            raise ValueError(f"No matching index found for {pred_str}")
        
        # Use this index to extract the corresponding predicted concentration from y_hat_reg
        pred_y_conc = 10 ** y_hat_reg[mis_index][i][pred_idx]
        
        # Append this predicted concentration to the sample's list
        sample_pred_y_conc.append(pred_y_conc)
    # Append the list of predicted concentrations for this sample to the main list
    pred_y_conc_list.append(sample_pred_y_conc)
    
# Convert the list to a more suitable data structure for your needs, such as a NumPy array or Pandas DataFrame

mis_df = pd.DataFrame({'True_Label':true_y, 'True_Conc':true_y_conc,'Pred_Label': pred_y, 'Pred_Conc':pred_y_conc_list})
mis_df.iloc[:,:]

In [ ]:
if not INFERENCE_MODE:
    pd.DataFrame(mis_df).to_csv(RESULT_PATH + "/misclassification_df_500epochs.csv",index=False)

In [ ]:

# Assuming y_true_label, y_pred_label, y_true_conc, y_hat_reg, and index are defined as per your context

all_classifications_list = []

# Loop through each sample in the dataset
for i in range(len(y_true_label)):
    true_label = y_true_label[i]
    pred_label = y_pred_label[i]
    true_concentration = y_true_conc[i]
    pred_concentration_list = []

    # Check if the prediction is a misclassification
    is_misclassified = true_label != pred_label

    # Processing predicted labels and concentrations
    preds_list = ast.literal_eval(pred_label)
    for pred in preds_list:
        pred_str = str([pred])
        pred_idx = [j for j, x in enumerate(index) if x == pred_str]

        if len(pred_idx) > 0:
            pred_idx = pred_idx[0]
        else:
            raise ValueError(f"No matching index found for {pred_str}")

        pred_concentration = 10 ** y_hat_reg[i][pred_idx]
        pred_concentration_list.append(pred_concentration)

    # Add the information to the list
    all_classifications_list.append({
        'True_Label': true_label, 
        'True_Conc': true_concentration, 
        'Pred_Label': pred_label, 
        'Pred_Conc': pred_concentration_list,
        'Misclassified': is_misclassified
    })

# Convert the list to a DataFrame
classification_df = pd.DataFrame(all_classifications_list)

# Display the DataFrame
classification_df

In [ ]:
# 5.2 Group and Calculate Accuracy
# Assuming classification_df is the DataFrame containing all samples

# Group the DataFrame by 'True_Label' and 'True_Conc'
grouped = classification_df.groupby(['True_Label', 'True_Conc'])

# Initialize dictionary to hold accuracy results
accuracy_dict = {}

# Loop through the groups
for (label, conc), group in grouped:
    # Total number of samples in the group
    total_samples = len(group)
    
    # Number of correctly classified samples
    correct_classifications = len(group[group['True_Label'] == group['Pred_Label']])
    
    # Calculate accuracy
    accuracy = correct_classifications / total_samples

    # Store the results
    accuracy_dict[(label, conc)] = {'Total': total_samples, 'Correct': correct_classifications, 'Accuracy': accuracy}

# Convert the dictionary to a DataFrame for better visualization
accuracy_df = pd.DataFrame.from_dict(accuracy_dict, orient='index')

# Display the DataFrame
accuracy_df

In [ ]:
if not INFERENCE_MODE:
    pd.DataFrame(accuracy_df).to_csv(RESULT_PATH + "/accuracy_by_conc_df_500epochs.csv",index=True)

In [ ]:
true_label = np.unique(mis_df['True_Label'])
# true_label

In [ ]:
print(mis_df.groupby(['True_Label','Pred_Label'], group_keys=False).apply(lambda x: x.shape))

#### Regression

In [ ]:
def calculate_MAE_and_R_square(real, predicted):
    MAE = np.abs(real-predicted).sum()/predicted.shape[0]
    y_mean = np.mean(real)
    ess = np.sum((predicted - real)**2)
    tss = np.sum((real - y_mean)**2)
    R = 1 - ess / tss
    return MAE, R

In [ ]:
summary_list = [] # Used for LOD determination, contains all the  'Virus_Type', 'Real_Concentration', 'Predicted_Concentration'

In [ ]:
## single
single_y_test_reg = y_test_reg.values[single_index,:]
single_y_hat_reg = y_hat_reg[single_index,:]
single_true_label = y_true_label[single_index]
cls = np.unique(single_true_label)

MAE1_list = []
R1_list = []

for i in range(9):
    
    plt.figure(figsize=(5, 5))
    single = ast.literal_eval(cls[i])
    ind = np.where(y_true_label==cls[i])
    
    single_y_test_reg = y_test_reg.values[ind,:][0]
    single_y_hat_reg = y_hat_reg[ind,:][0]
    not_outlier = np.where((single_y_hat_reg[:, i] > -25) & (single_y_hat_reg[:, i] < 8))
    real = single_y_test_reg[:,i][not_outlier]
    predicted = single_y_hat_reg[:,i][not_outlier]
    
    c = np.nonzero(np.all(single_y_test_reg != 0, axis=0))[0]
    MAE1, R1 = calculate_MAE_and_R_square(single_y_test_reg[:,c], single_y_hat_reg[:,c])
    MAE1_list.append(MAE1)
    R1_list.append(R1)
    
    # Loop through each element in the concentration arrays
    virus_type = mlb1.classes_[i]
    summary_dict = {
        'Virus_Type': virus_type,
        'Real_Concentration': real,
        'Predicted_Concentration': predicted
    }
    summary_list.append(summary_dict)  

    # Save data to CSV - add this block
    df_virus = pd.DataFrame({
        'True_Conc': real,
        'Pred_Conc': predicted
    })
    if not INFERENCE_MODE:   
        df_virus.to_csv(RESULT_PATH + f"/regression_data_{virus_type}_500epochs.csv", index=False)
    
    plt.figure(figsize=(5,5))
    plt.scatter(real, predicted,color="blue",edgecolors="grey",alpha = 0.3, linewidths=0.1)
   
    plt.axline((1.5,1.5),(5.1,5.1),linestyle='--', color="grey")
    plt.text(x=3.6,y=2.6,s="MAE_"+single[0]+" = " +str(round(MAE1,4)),style='oblique')
    plt.text(x=3.6,y=1.7,s="R^2_"+single[0]+" = " +str(round(R1,4)),style='oblique')

    plt.xlabel("Actual log10(C)")
    plt.ylabel("Predicted log10(C)")
    plt.title(virus_type)

    if not INFERENCE_MODE:
        plt.savefig(RESULT_PATH + f"/regression_{virus_type}_500epochs.png")
    plt.show()
    

for item in MAE1_list:
    print(item.round(4))
print()
for item in R1_list:
    print(item.round(4))
print()

#### Unknown test with multiple csv folders (each folder has a sample csv)

In [ ]:
# ============================================================
# UNKNOWN TEST — Cell 1: Inference
# Run once per model. Saves raw per-spectrum probabilities and
# regression outputs so any gamma can be applied later cheaply.
# Prerequisites in scope: model, scaler1, mlb1,
# ============================================================
import os, re, ast
import numpy as np
import pandas as pd
from collections import Counter

# ── USER CONFIG ──────────────────────────────────────────────
UNKNOWN_ROOT     = "/scratch/jc76425/DNA_RNA_hybridization/data/03172026-Unknown_test_extracted_results"
WAVENUMBER_START = 400
WAVENUMBER_END   = 1800
DATE_STR         = "03172026"
UNKNOWN_RESULT_PATH = (
    f"/scratch/jc76425/DNA_RNA_hybridization/results/"
    f"{DATE_STR}-Extracted_unknown_test_weight_{CLASS_WEIGHT}vs{REG_WEIGHT}"
)
os.makedirs(UNKNOWN_RESULT_PATH, exist_ok=True)
# ─────────────────────────────────────────────────────────────

CLASS_NAMES = list(mlb1.classes_)   # e.g. ['B1', 'B1351', ...]
N_CLASSES   = len(CLASS_NAMES)
wn_cols     = [str(w) for w in range(WAVENUMBER_START, WAVENUMBER_END + 1)]

_folder_re = re.compile(r'^sample\d+_(.+?)-(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$')

def parse_folder(folder_name):
    m = _folder_re.match(folder_name)
    if m is None:
        return None, None, None
    return m.group(1), float(m.group(2)), float(m.group(3))

def load_and_split_ml_csv(folder_path, conc_high, conc_low):
    """
    Load the single combined ML_format CSV, split rows by Conc,
    scale with scaler1, return (X_reshaped, conc_array).
    """
    ml_csvs = [
        f for f in os.listdir(folder_path)
        if f.endswith('ML_format.csv') and 'extracted' in f.lower()
    ]
    if len(ml_csvs) == 0:
        raise FileNotFoundError(f"No ML_format CSV found in {folder_path}")

    df = pd.read_csv(os.path.join(folder_path, ml_csvs[0]))
    df['_conc'] = df['Conc'].apply(lambda v: float(str(v).strip('[]')))
    cols_present = [c for c in wn_cols if c in df.columns]

    all_X, all_concs = [], []
    for conc_val in [conc_high, conc_low]:
        subset = df[df['_conc'] == conc_val]
        if len(subset) == 0:
            continue
        X          = subset[cols_present].values.astype(np.float32)
        X_scaled   = scaler1.transform(X)
        X_reshaped = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)
        all_X.append(X_reshaped)
        all_concs.append(np.full(len(subset), conc_val, dtype=np.float64))

    if not all_X:
        raise ValueError(
            f"No rows matched conc_high={conc_high} or conc_low={conc_low}. "
            f"Found: {sorted(df['_conc'].unique())}"
        )
    return np.concatenate(all_X, axis=0), np.concatenate(all_concs, axis=0)

# ── Run inference on all 88 samples ──────────────────────────
sample_folders = sorted([
    d for d in os.listdir(UNKNOWN_ROOT)
    if os.path.isdir(os.path.join(UNKNOWN_ROOT, d)) and d.startswith('sample')
])

spectrum_rows = []

for folder in sample_folders:
    folder_path = os.path.join(UNKNOWN_ROOT, folder)
    virus, conc_high, conc_low = parse_folder(folder)
    if virus is None:
        print(f"[WARN] Could not parse: {folder}, skipping.")
        continue

    try:
        X_sample, conc_array = load_and_split_ml_csv(
            folder_path, conc_high, conc_low
        )
    except (FileNotFoundError, ValueError) as e:
        print(f"[WARN] {folder}: {e}, skipping.")
        continue

    prob_matrix, reg_matrix = model.predict(X_sample, verbose=0)

    for i in range(len(X_sample)):
        row = {
            'folder':         folder,
            'true_label':     f"['{virus}']",
            'true_conc_high': conc_high,
            'true_conc_low':  conc_low,
            'true_conc_this_spectrum': float(conc_array[i]),
            'spectrum_idx':   i,
        }
        for j, cls in enumerate(CLASS_NAMES):
            row[f'prob_{cls}'] = float(prob_matrix[i, j])
            row[f'reg_{cls}']  = float(reg_matrix[i, j])
        spectrum_rows.append(row)

spectrum_df = pd.DataFrame(spectrum_rows)
spectrum_df.to_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "spectrum_level_details.csv"),
    index=False
)
print(f"Inference complete. {len(sample_folders)} samples, "
      f"{len(spectrum_df)} total spectra.")
print(f"Saved: {UNKNOWN_RESULT_PATH}/spectrum_level_details.csv")

In [ ]:
# ============================================================
# UNKNOWN TEST — Cell 2: Evaluation function
# Reconstructs sample-level results from saved spectrum details
# at any gamma, without re-running inference.
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def evaluate_at_gamma(spectrum_df, gamma, CLASS_NAMES,
                      UNKNOWN_RESULT_PATH, save=True):
    """
    Apply threshold gamma to saved raw probabilities and produce:
      - Sample-level results DataFrame
      - Confusion matrix (raw counts + normalized), saved as PNG and CSV
      - Regression scatter plots per virus, saved as PNG
      - Summary CSVs

    Parameters
    ----------
    spectrum_df       : DataFrame loaded from spectrum_level_details.csv
    gamma             : float, classification threshold in [0, 1]
    CLASS_NAMES       : list of class name strings from mlb1.classes_
    UNKNOWN_RESULT_PATH : str, root output folder for this model
    save              : bool, whether to write files to disk

    Returns
    -------
    results_df : sample-level DataFrame
    """
    prob_cols = [f'prob_{c}' for c in CLASS_NAMES]
    reg_cols  = [f'reg_{c}'  for c in CLASS_NAMES]

    prob_matrix = spectrum_df[prob_cols].values
    reg_matrix  = spectrum_df[reg_cols].values
    pred_binary = (prob_matrix >= gamma).astype(int)

    # Rebuild per-spectrum predicted label strings
    pred_labels = []
    for row in pred_binary:
        active = [CLASS_NAMES[j] for j in range(len(CLASS_NAMES)) if row[j] == 1]
        pred_labels.append(str(active) if active else "[]")
    spectrum_df = spectrum_df.copy()
    spectrum_df['pred_label'] = pred_labels

    # ── Sample-level aggregation ──────────────────────────────
    sample_rows = []
    for folder, grp in spectrum_df.groupby('folder', sort=False):
        true_label_str = grp['true_label'].iloc[0]
        conc_high      = grp['true_conc_high'].iloc[0]
        conc_low       = grp['true_conc_low'].iloc[0]
        virus          = ast.literal_eval(true_label_str)[0]

        try:
            true_class_idx = CLASS_NAMES.index(virus)
        except ValueError:
            true_class_idx = None

        grp_indices      = grp.index
        correct_mask     = (grp['pred_label'] == true_label_str).values
        above_gamma_mask = (pred_binary[grp_indices].sum(axis=1) > 0)

        # Predicted concentration: mean over correctly labelled spectra
        if correct_mask.sum() > 0 and true_class_idx is not None:
            reg_vals         = reg_matrix[grp_indices[correct_mask], true_class_idx]
            sample_pred_conc = np.mean(10 ** reg_vals)
        else:
            sample_pred_conc = np.nan

        # Predicted label: majority vote among above-gamma spectra
        if above_gamma_mask.sum() > 0:
            sample_pred_label = Counter(
                grp.loc[grp_indices[above_gamma_mask], 'pred_label']
            ).most_common(1)[0][0]
        else:
            sample_pred_label = "[]"

        sample_rows.append({
            'folder':                folder,
            'true_label':            true_label_str,
            'true_conc_high':        conc_high,
            'true_conc_low':         conc_low,
            'pred_label':            sample_pred_label,
            'pred_conc':             sample_pred_conc,
            'n_spectra_total':       len(grp),
            'n_spectra_above_gamma': int(above_gamma_mask.sum()),
            'n_spectra_correct':     int(correct_mask.sum()),
        })

    results_df = pd.DataFrame(sample_rows)
    n_correct  = (results_df['true_label'] == results_df['pred_label']).sum()
    n_total    = len(results_df)
    print(f"γ={gamma:.2f} | Sample accuracy: {n_correct}/{n_total} "
          f"({100*n_correct/n_total:.1f}%)")

    if not save:
        return results_df

    # ── Output folder for this gamma ─────────────────────────
    gamma_path = os.path.join(UNKNOWN_RESULT_PATH, f"gamma_{gamma:.2f}")
    os.makedirs(gamma_path, exist_ok=True)

    results_df.to_csv(
        os.path.join(gamma_path, "sample_level_results.csv"), index=False
    )

    # ── Confusion matrix ─────────────────────────────────────
    def get_text_color(val, min_val, max_val):
        return 'black' if val >= (max_val - min_val) / 2 else 'white'

    y_true    = results_df['true_label'].values
    y_pred    = results_df['pred_label'].values
    cm_labels = sorted(
        np.unique(np.concatenate([y_true, y_pred])),
        key=lambda x: (x.count(','), x)
    )
    cm        = confusion_matrix(y_true, y_pred, labels=cm_labels)
    cm_df     = pd.DataFrame(cm, index=cm_labels, columns=cm_labels)
    fig_size  = max(8, len(cm_labels))

    # Raw counts
    plt.figure(figsize=(fig_size, fig_size))
    ax = sns.heatmap(cm_df, annot=False, fmt='d', cmap='Blues_r',
                     linewidths=2, linecolor='black')
    min_v, max_v = cm.min(), cm.max()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j] != 0:
                ax.text(j+0.5, i+0.5, cm[i, j], fontsize=14,
                        ha='center', va='center',
                        color=get_text_color(cm[i, j], min_v, max_v))
    ax.collections[0].colorbar.remove()
    plt.xlabel("Predicted label"); plt.ylabel("True label")
    plt.title(f"Sample-level Confusion Matrix (γ={gamma:.2f})")
    plt.tight_layout()
    plt.savefig(os.path.join(gamma_path, "confusion_matrix.png"), dpi=150)
    plt.show()
    cm_df.to_csv(os.path.join(gamma_path, "confusion_matrix.csv"))

    # Normalized
    cm_norm    = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm_df = pd.DataFrame(cm_norm, index=cm_labels, columns=cm_labels)
    plt.figure(figsize=(fig_size, fig_size))
    ax = sns.heatmap(cm_norm_df, annot=False, cmap='Blues_r',
                     linewidths=2, linecolor='black', vmin=0, vmax=1)
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            if cm_norm[i, j] > 0:
                color = 'black' if cm_norm[i, j] >= 0.5 else 'white'
                ax.text(j+0.5, i+0.5, f"{cm_norm[i, j]:.2f}", fontsize=14,
                        ha='center', va='center', color=color)
    ax.collections[0].colorbar.remove()
    plt.xlabel("Predicted label"); plt.ylabel("True label")
    plt.title(f"Sample-level Confusion Matrix — Normalized (γ={gamma:.2f})")
    plt.tight_layout()
    plt.savefig(os.path.join(gamma_path, "confusion_matrix_normalized.png"), dpi=150)
    plt.show()
    cm_norm_df.to_csv(os.path.join(gamma_path, "confusion_matrix_normalized.csv"))

    # ── Regression scatter plots ──────────────────────────────
    correctly_predicted = results_df[
        results_df['true_label'] == results_df['pred_label']
    ].dropna(subset=['pred_conc'])

    for virus_label in sorted(correctly_predicted['true_label'].unique()):
        subset = correctly_predicted[correctly_predicted['true_label'] == virus_label]
        if subset.empty:
            continue
        real_log      = np.log10(subset['true_conc_high'].values + 1e-20)
        predicted_log = np.log10(subset['pred_conc'].values      + 1e-20)
        MAE    = np.mean(np.abs(real_log - predicted_log))
        ss_res = np.sum((real_log - predicted_log) ** 2)
        ss_tot = np.sum((real_log - real_log.mean()) ** 2)
        R2     = 1 - ss_res / ss_tot if ss_tot > 1e-12 else float('nan')
        virus_name = ast.literal_eval(virus_label)[0]

        plt.figure(figsize=(5, 5))
        plt.scatter(real_log, predicted_log, color='blue',
                    edgecolors='grey', alpha=0.6, linewidths=0.5)
        lo = min(real_log.min(), predicted_log.min()) - 0.3
        hi = max(real_log.max(), predicted_log.max()) + 0.3
        plt.plot([lo, hi], [lo, hi], linestyle='--', color='grey')
        plt.text(lo + 0.1*(hi-lo), hi - 0.15*(hi-lo),
                 f"MAE = {MAE:.4f}", style='oblique', fontsize=10)
        plt.text(lo + 0.1*(hi-lo), hi - 0.25*(hi-lo),
                 f"R² = {R2:.4f}", style='oblique', fontsize=10)
        plt.xlabel("Actual log₁₀(C)")
        plt.ylabel("Predicted log₁₀(C)")
        plt.title(f"{virus_name} — sample-level regression (γ={gamma:.2f})")
        plt.tight_layout()
        plt.savefig(os.path.join(gamma_path, f"regression_{virus_name}.png"), dpi=150)
        plt.show()

    correctly_predicted[['true_label', 'true_conc_high', 'pred_conc']].to_csv(
        os.path.join(gamma_path, "regression_summary.csv"), index=False
    )

    return results_df

In [ ]:
# ============================================================
# UNKNOWN TEST — Cell 3: Apply gamma sweep
# Re-run this cell freely with any gammas — no model inference.
# ============================================================

spectrum_df = pd.read_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "spectrum_level_details.csv")
)

for gamma in [0.5, 0.6, 0.7, 0.8, 0.9]:
    results_df = evaluate_at_gamma(
        spectrum_df, gamma, CLASS_NAMES, UNKNOWN_RESULT_PATH, save=True
    )